In [20]:
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader

# 1. Data Preparation and Loading

In [21]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [22]:
train_dataset = CIFAR10(
    root="./data",
    train=True, 
    download=True,
    transform=transform
)

In [23]:
batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
)

In [24]:
print(f"Número de imágenes: {len(train_dataset)}")
print(f"Número de clases: {len(train_dataset.classes)}")
print(f"Clases: {train_dataset.classes}")

Número de imágenes: 50000
Número de clases: 10
Clases: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [25]:
# Verify that the images are normalized correctly
real_images, labels = next(iter(train_loader))

print(real_images.shape)
print(real_images.min())
print(real_images.max())

torch.Size([64, 3, 32, 32])
tensor(-1.)
tensor(1.)


# 2. Model Architecture
## 2.1 Latent space

In [26]:
z_dim = 128

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

z = torch.randn(batch_size, z_dim, device=device)

In [27]:
print(z.shape)

torch.Size([64, 128])


## 2.2 Generator

In [28]:
class Generator(nn.Module):
    def __init__(self, latent_dim=128, dim=128):
        super().__init__()

        self.latent_dim = latent_dim
        self.dim = dim

        self.linear =  nn.Linear(latent_dim, 4 * 4 * 4 * dim)

        self.bn1 = nn.BatchNorm1d(4 * 4 * 4 * dim)

        self.deconv1 = nn.ConvTranspose2d(4 * dim, 2 * dim, kernel_size=5, stride=2, padding=2, output_padding=1)

        self.bn2 = nn.BatchNorm2d(2 * dim)

        self.deconv2 = nn.ConvTranspose2d(2 * dim, dim, kernel_size=5, stride=2, padding=2, output_padding=1)

        self.bn3 = nn.BatchNorm2d(dim)

        self.deconv3 = nn.ConvTranspose2d(dim, 3, kernel_size=5, stride=2, padding=2, output_padding=1)

        self.relu = nn.ReLU()

        self.tanh = nn.Tanh()

    def forward(self, z):
        # Linear transformation
        x = self.linear(z)

        # Batch normalization + ReLU activation
        x = self.bn1(x)
        x = self.relu(x)

        # Reshape into a 4x4 feature map
        x = x.view(-1, 4 * self.dim, 4, 4)

        # 4x4 -> 8x8
        x = self.deconv1(x)
        x = self.bn2(x)
        x = self.relu(x)

        # 8x8 -> 16x16
        x = self.deconv2(x)
        x = self.bn3(x)
        x = self.relu(x)

        # 16x16 -> 32x32
        x = self.deconv3(x)

        # Output activation
        x = self.tanh(x)

        return x

## 2.3 Critic

In [29]:
class Critic(nn.Module):
    def __init__(self, dim=128):
        super().__init__()

        self.dim = dim

        self.conv1 = nn.Conv2d(3, dim, kernel_size=5, stride=2, padding=2)

        self.leaky_relu = nn.LeakyReLU(0.2)

        self.conv2 = nn.Conv2d(dim, 2 * dim, kernel_size=5, stride=2, padding=2)

        self.bn2 = nn.BatchNorm2d(2 * dim)

        self.conv3 = nn.Conv2d(2 * dim, 4 * dim, kernel_size=5, stride=2, padding=2)

        self.bn3 = nn.BatchNorm2d(4 * dim)

        self.linear = nn.Linear(4 * 4 * 4 * dim, 1)

    def forward(self, x):
        # 32x32 -> 16x16
        x = self.conv1(x)
        x = self.leaky_relu(x)

        # 16x16 -> 8x8
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.leaky_relu(x)

        # 8x8 -> 4x4
        x = self.conv3(x)
        x = self.bn3(x)
        x = self.leaky_relu(x)

        # Flatten
        x = x.view(-1, 4 * 4 * 4 * self.dim)

        # Linear transformation
        x = self.linear(x)

        return x

# 3. WGAN

In [30]:
def train_critic(generator, critic, real_images, optimizer_D, z_dim, device):
    batch_size = real_images.size(0)

    # Generate fake images
    z = torch.randn(batch_size, z_dim, device=device)
    fake_images = generator(z)

    # Compute Wasserstein loss
    real_scores = critic(real_images)
    fake_scores = critic(fake_images.detach())
    critic_loss = torch.mean(fake_scores) - torch.mean(real_scores)

    # Update critic parameters
    optimizer_D.zero_grad()
    critic_loss.backward()
    optimizer_D.step()

    # Weight clipping
    for p in critic.parameters():
        p.data.clamp_(-0.01, 0.01)

    return critic_loss.item()

def train_generator(generator, critic, optimizer_G, batch_size, z_dim, device):
    z = torch.randn(batch_size, z_dim, device=device)

    fake_images = generator(z)

    # Compute generator loss
    fake_scores = critic(fake_images)
    generator_loss = -torch.mean(fake_scores)

    # Update generator parameters
    optimizer_G.zero_grad()
    generator_loss.backward()
    optimizer_G.step()

    return generator_loss.item()

In [ ]:
from itertools import cycle

CRITIC_ITERATIONS = 5
N_ITERATIONS = 1000 #200000

generator = Generator(latent_dim=z_dim, dim=128).to(device)
critic = Critic(dim=128).to(device)

optimizer_G = torch.optim.RMSprop(generator.parameters(), lr=5e-5)
optimizer_D = torch.optim.RMSprop(critic.parameters(), lr=5e-5)

train_iterator = cycle(train_loader)

generator_losses = []
critic_losses = []

for iteration in range(N_ITERATIONS):
    for _ in range(CRITIC_ITERATIONS):
        real_images, _ = next(train_iterator)
        real_images = real_images.to(device)

        critic_loss = train_critic(generator, critic, real_images, optimizer_D, z_dim, device)
        critic_losses.append(critic_loss)

    generator_loss = train_generator(generator, critic, optimizer_G, batch_size, z_dim, device)
    generator_losses.append(generator_loss)

    if iteration % 100 == 0:
        print(f"Iteration {iteration}: Critic Loss: {critic_loss:.4f}, Generator Loss: {generator_loss:.4f}")

Iteration 0: Critic Loss: -0.1069, Generator Loss: 0.0875
Iteration 100: Critic Loss: -1.3683, Generator Loss: 0.7051
Iteration 200: Critic Loss: -1.3784, Generator Loss: 0.6774


In [ ]:
import matplotlib.pyplot as plt

generator.eval()

with torch.no_grad():
    z = torch.randn(64, z_dim, device=device)
    generated_images = generator(z)

generator.train()

generated_images = (generated_images + 1) / 2 # Rescale to [0, 1]

fig, axes = plt.subplots(8, 8, figsize=(8, 8))

for i, ax in enumerate(axes.flat):
    image = generated_images[i].cpu().permute(1, 2, 0)

    ax.imshow(image)
    ax.axis("off")

plt.tight_layout()
plt.show()

# 4. WGAN-GP
## 4.1 Interpolation
## 4.2 Gradient penalty
## 4.3 WGAN-GP critic loss
## 4.4 Training loop

# 5. Evaluation
## 5.1 Generated samples
## 5.2 Loss curves
## 5.3 FID / Inception Score
## 5.4 WGAN vs WGAN-GP